# Track C scope and the stuck-episode check (Oct 31, 2026) — descriptive, no model change

`results/model_b_log.md`, entry 4: the scope, the reading rule and the implementation choices were committed there (923f3b7) before any number below. Code: `scripts/p24_noOldCL.py` (simulation), `src/slider/stuck.py` (episodes).

**Spec (as given):**

> TASK: Track C scope + stuck-episode check. Descriptive only; NO model change. Never load
> Ring/CT.
>
> 0. FIRST, log entry 4 (results/model_b_log.md), committed before computing anything:
>    - Scope (declared Oct 31, before Ring/CT): Track C is evaluated on v09 and cloud rounds
>      only; old-CL rounds (0.02 step) are out of scope. All model comparisons (humans,
>      L1, final, C2, later APEM) use the same rounds.
>    - Reading rule for the check below (fixed now):
>      (i)  C2's median time to first try exceeds 1.5x the human value, and lies outside
>           the human CI, in >= 2 episode types -> candidate fix: a stuck feature in the
>           try hazard;
>      (ii) else, if C2's tries-until-break exceed 1.5x the human value in Full standoffs
>           only -> candidate fix: standoff-specific keep values;
>      (iii) else -> freeze C2.
>
> 1. RECOMPUTE WITHOUT OLD CL
>    - Human targets 1-10 and their CIs for Full CL and Hierarchy CL on v09 rounds only
>      (same code and bootstrap as the target tables). The other cells are unchanged.
>    - Simulate L1, final and C2 on only those v09 rounds, R = 200 (seeds 1101, 4101, 8101;
>      register them).
>    - Rebuild the pass/miss and gap tables with these two cells replaced, and print
>      before/after for those cells. Save as results/tables/p24_checks_noOldCL.csv.
>
> 2. STUCK EPISODES (G-train humans vs C2 on the G6train design minus old-CL rounds;
>    10 C2 replicates, seed 8001; per interface: v09, CC)
>    - Episode = all 7 players at a wall (|s| = 1), not consensus, lasting >= 0.5 s.
>      It ends when any player reaches the opposite wall, or at round end.
>    - Types: Full/Star split k-(7-k) (6-1, 5-2, 4-3); Hierarchy fraught (the fraught
>      test at the corner); Hierarchy non-fraught.
>    - Per type, report:
>        episodes per round; median duration; median time to first try;
>        median tries until the episode ends;
>        P(kept) per try (B2 detector measure, the same on both);
>        for Full: share of episodes broken by the minority.
>    - Human values with group-bootstrap 95% CIs (1,000, stratified, seed 573).
>    - Print one table: human vs C2 side by side, and the reading rule's verdict.
>
> Notebook notebooks/path1/pC2_stuck.ipynb (spec cell first, short cells). Execute, add a
> README line, commit, report. Then stop.

## Choices (in short; the full list is in entry 4)

- **Scope.** Track C is evaluated on v09 and CC rounds only; old-CL rounds are out.
- **Part 1.** Full CL and Hierarchy CL keep only their v09 rounds: human targets 1-10 with the target-table code (10,000 resamples, seed 573) and the zero-width rule; L1, final and C2 simulated on those rounds (`p24_noOldCL.py`, R = 200, seeds 1101, 4101, 8101). The other cells keep the main runs and the saved human values. `p24_checks_noOldCL.csv`: L1, final, C2, the `p24_checks.csv` rows plus `half_width` and `gap`.
- **Part 2.** Humans: G-train (G6train run set) minus old CL, minus runs with a bot seat. C2: the G6train design minus old CL, R = 10, seed 8001. Episodes, types, tries and statistics as in `stuck.py`; human CIs from 1,000 file resamples within dataset, seed 573.
- **Reading rule.** Cell = interface x topology x type. (i): a type (6-1, 5-2, 4-3, fraught, non-fraught) counts if C2's median time to first try is > 1.5 x human and outside the human CI in one of its cells; fires at >= 2 types. (ii): C2's median tries > 1.5 x human in >= 1 Full split cell and in no other cell. (iii): else freeze C2.


In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

from slider import config, design, interface, stuck, targets

sys.path.insert(0, str(config.ROOT / "scripts"))
import p24_checks as pc                                                   # noqa: E402
import p24_noOldCL                                                        # noqa: E402

warnings.filterwarnings("ignore", message="Pandas requires version")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

PRESETS = ["L1", "final", "C2"]
CL_CELLS = [("Full", "CL"), ("Hierarchy", "CL")]
KEY = pc.KEY


def v09_cl(runs, table):
    """The CL v09 runs of a run set."""
    keep = ((table["dataset"] == "CL") & [interface.version_tag(f) == "v09" for f in table["file"]]).to_numpy()
    return [r for r, k in zip(runs, keep) if k], table[keep].reset_index(drop=True)


r3, t3 = v09_cl(*design.load("L3"))
r5, t5 = v09_cl(*design.load("L5"))
pd.DataFrame({"L3 rounds": t3.groupby("topology").size(), "L5 rounds": t5.groupby("topology").size(),
              "groups": t3.groupby("topology")["file"].nunique()})


/tmp/ipykernel_13282/425812756.py:25: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  keep = ((table["dataset"] == "CL") & [interface.version_tag(f) == "v09" for f in table["file"]]).to_numpy()


/tmp/ipykernel_13282/425812756.py:25: FutureWarning: Logical ops (and, or, xor) between Pandas objects and dtype-less sequences (e.g. list, tuple) are deprecated and will raise in a future version. Wrap the object in a Series, Index, or np.array before operating instead.
  keep = ((table["dataset"] == "CL") & [interface.version_tag(f) == "v09" for f in table["file"]]).to_numpy()


,L3 rounds,L5 rounds,groups
topology,,,
Full,39,39,13
Hierarchy,39,39,13


**How to read it.** The v09 rounds of the two CL cells: L3 (targets 1-7) and L5 (targets 8-10, bot-free, complete) rounds, and log files.

## Part 1 — targets without old CL

In [2]:
# human targets 1-10 on the v09 CL runs, with the code of the saved target tables
tt = targets.train_targets(r3, t3)
tt = tt[tt["role"] == "train+val"]
parts = [tt[tt["stat"] == targets.CHECKS[t]].assign(target=t) for t in range(1, 8)]
parts.append(targets.reversal_rate_table(r5).rename(columns={"rate": "value"}).assign(target=8, stat=targets.CHECKS[8]))
ag = targets.agreement_at_reversal_table(r5)[0]
parts.append(ag[(ag["measure"] == "f_rev") & (ag["stat"] == "median")].assign(target=9, stat=targets.CHECKS[9]))
cs = targets.crossing_time_table(r5)[0]
parts += [cs[cs["stat"] == s].assign(target=10) for s in [targets.CHECKS[10], targets.PAUSE_SHARE]]
new = pd.concat(parts)[KEY + ["value", "ci_lo", "ci_hi"]].reset_index(drop=True)
new["in_check_list"] = (new["stat"] != targets.PAUSE_SHARE) & ((new["target"] != 3) | (new["topology"] == "Hierarchy"))
new = new[new["in_check_list"] | (new["stat"] == targets.PAUSE_SHARE)].reset_index(drop=True)

old = targets.human_check_values()
old = old[old["in_check_list"] | (old["stat"] == targets.PAUSE_SHARE)]
is_cl = lambda d: (d["dataset"] == "CL").to_numpy()
a, b = set(map(tuple, old.loc[is_cl(old), KEY].values)), set(map(tuple, new[KEY].values))
assert a == b, (a - b, b - a)
n_rounds = pc.rounds_per_cell()
n_rounds.update({(t, "CL"): n for t, n in zip(tt.loc[tt["stat"] == targets.CHECKS[1], "topology"],
                                               tt.loc[tt["stat"] == targets.CHECKS[1], "n_runs"])})
human = {"before": pc.zero_width_rule(old, pc.rounds_per_cell()),
         "after": pc.zero_width_rule(pd.concat([old[~is_cl(old)], new], ignore_index=True), n_rounds)}

# the between-group SD of the per-group reversal rates (L3 trigger rows), on the v09 CL runs
g = targets.group_rates(targets.reversal_rate_runs(r5))
sd_new = []
for (topology, dataset), c in g.groupby(["topology", "dataset"], sort=False):
    lo, hi = targets.between_group_sd_ci(c["rate"].to_numpy())
    sd_new.append({"target": 8, "stat": pc.SD_STAT, "topology": topology, "dataset": dataset,
                   "value": targets.between_group_sd(c["rate"]), "ci_lo": lo, "ci_hi": hi, "in_check_list": False, "ci_rule": ""})
sd_old = pc.human_sd()
sd = {"before": sd_old, "after": pd.concat([sd_old[~is_cl(sd_old)], pd.DataFrame(sd_new)], ignore_index=True)}
print("zero-width rule applied (after):", human["after"].loc[human["after"]["ci_rule"] != "", KEY + ["ci_rule"]].values.tolist())


zero-width rule applied (after): [[1, 'success_rate_pct', 'Star', 'CC', 'Clopper-Pearson'], [10, 'crossing_time_median_s', 'Full', 'CC', 'lab clock +/- 0.05 s'], [10, 'crossing_time_median_s', 'Star', 'CC', 'lab clock +/- 0.05 s']]


In [3]:
# replicates: the main runs, with the CL cells replaced by the v09-only runs
reps = {"before": {}, "after": {}}
for p in PRESETS:
    main = pd.read_parquet(config.P24_REPS[p])
    v09 = pd.read_parquet(p24_noOldCL.path(p))
    assert main["replicate"].nunique() == v09["replicate"].nunique() == 200
    reps["before"][p] = main
    reps["after"][p] = pd.concat([main[~is_cl(main)], v09[is_cl(v09)]], ignore_index=True)
assert set(v09["dataset"]) == {"CL"}

tables = {}
for when in ["before", "after"]:
    t = pd.concat(pc.build_tables(reps[when], human[when], sd[when]).values(), ignore_index=True)
    t["half_width"] = (t["ci_hi"] - t["ci_lo"]) / 2
    t["gap"] = (t["sim_mean"] - t["value"]) / t["half_width"]
    tables[when] = t.rename(columns={"value": "human"})
before = pd.read_csv(config.P24_CHECKS)
b = tables["before"].set_index(["preset"] + KEY)
assert np.allclose(b.loc[before.set_index(["preset"] + KEY).loc[PRESETS].index, "sim_mean"],
                   before.set_index(["preset"] + KEY).loc[PRESETS, "sim_mean"], equal_nan=True)   # "before" = p24_checks.csv
cols = ["preset"] + KEY + ["in_check_list", "R", "n", "sim_mean", "sim_sd", "se_mc", "human", "ci_lo", "ci_hi", "ci_rule",
        "verdict", "borderline", "half_width", "gap"]
out = tables["after"][cols]
out.to_csv(config.TABLES / "p24_checks_noOldCL.csv", index=False)
print(len(out), "rows saved to p24_checks_noOldCL.csv")


195 rows saved to p24_checks_noOldCL.csv


In [4]:
def fmt(x, small):
    return f"{x:.3f}" if small else f"{x:.2f}"


rows = []
show = lambda t: t[(t["dataset"] == "CL") & (t["in_check_list"] | (t["stat"] == targets.PAUSE_SHARE))]
B, A = show(tables["before"]).set_index(["preset"] + KEY), show(tables["after"]).set_index(["preset"] + KEY)
for key in A.loc["L1"].index:
    hb, ha = B.loc[("L1",) + key], A.loc[("L1",) + key]
    small = abs(hb["human"]) < 1
    row = {"target": key[0], "stat": key[1], "cell": f"{key[2]} {key[3]}",
           "human before": f"{fmt(hb['human'], small)} [{fmt(hb['ci_lo'], small)}, {fmt(hb['ci_hi'], small)}]",
           "human after": f"{fmt(ha['human'], small)} [{fmt(ha['ci_lo'], small)}, {fmt(ha['ci_hi'], small)}]"}
    for p in PRESETS:
        for when, T in [("before", B), ("after", A)]:
            r = T.loc[(p,) + key]
            row[f"{p} {when}"] = f"{fmt(r['sim_mean'], small)} ({r['gap']:+.2f} {'P' if r['verdict'] == 'pass' else 'M'})"
    rows.append(row)
print(pd.DataFrame(rows).sort_values(["target", "cell"], kind="stable").to_string(index=False))


 target                       stat         cell          human before           human after        L1 before         L1 after      final before      final after        C2 before         C2 after
      1           success_rate_pct      Full CL  82.54 [69.84, 92.06]  89.74 [82.05, 97.44]  40.31 (-3.80 M)  61.31 (-3.70 M)   33.60 (-4.40 M)  51.03 (-5.03 M)  51.82 (-2.76 M)  77.73 (-1.56 M)
      1           success_rate_pct Hierarchy CL 95.08 [89.47, 100.00] 97.44 [92.31, 100.00]  63.06 (-6.08 M)  87.64 (-2.55 M)   58.96 (-6.86 M)  82.36 (-3.92 M)  72.50 (-4.29 M)  97.50 (+0.02 P)
      2 consensus_time_km_median_s      Full CL 89.19 [71.09, 118.70] 72.31 [58.01, 106.17] 299.40 (+8.83 M) 224.15 (+6.31 M)  300.00 (+8.86 M) 268.94 (+8.17 M) 269.50 (+7.57 M) 149.54 (+3.21 M)
      2 consensus_time_km_median_s Hierarchy CL  47.06 [36.84, 67.74]  36.84 [21.41, 45.10] 181.30 (+8.69 M)  98.31 (+5.19 M) 214.54 (+10.84 M) 112.66 (+6.40 M) 110.31 (+4.09 M)  63.05 (+2.21 M)
      3           fraught

**How to read it.** The two CL cells, before (all CL rounds, `p24_checks.csv`) and after (v09 rounds only). Human: value [95% CI]. Each model: its mean over R = 200 replicates (gap in CI half-widths, P = pass, M = miss). Pass / miss as in P24: the mean inside the human CI.

In [5]:
summary = []
for when in ["before", "after"]:
    t = tables[when][tables[when]["in_check_list"]]
    for p in PRESETS:
        c = t[t["preset"] == p]
        summary.append({"preset": p, "when": when, "passes (of 53)": int((c["verdict"] == "pass").sum()),
                        "median |gap|": c["gap"].abs().median(),
                        "CL passes (of 20)": int((c.loc[c["dataset"] == "CL", "verdict"] == "pass").sum())})
pd.DataFrame(summary).pivot(index="preset", columns="when").reindex(PRESETS).round(2)


passes (of 53)        median |gap|        CL passes (of 20)       
when            after before        after before             after before
preset                                                                   
L1                 17     17         1.60   1.80                 4      4
final              16     16         1.69   1.86                 4      4
C2                 30     31         0.86   0.74                 7      8

**How to read it.** Over the 53 check statistics, per model: passes and median |gap|, before and after; the last pair counts the passes among the 20 statistics of the two CL cells.

## Part 2 — stuck episodes (G-train humans vs. C2)

In [6]:
runs_h, table_h = design.load("G6train")                               # G-train only
assert set(table_h["role"]) == {"train"} and set(table_h["topology"]) <= {"Full", "Hierarchy", "Star"}
iface_h = np.where(table_h["dataset"] == "CC", "cc", ["lab_v09" if interface.version_tag(f) == "v09" else "old_cl"
                                                       for f in table_h["file"]])
keep = (iface_h != "old_cl") & ~table_h["has_bot"].to_numpy()
runs_h = [r for r, k in zip(runs_h, keep) if k]
table_h, iface_h = table_h[keep].reset_index(drop=True), iface_h[keep]

d6 = design.design("G6train")
d6 = d6[d6["interface"] != "old_cl"].reset_index(drop=True)
R6 = 10
runs_c, table_c = design.simulate_design(d6, "C2", R=R6, seed=8001)
iface_c = np.repeat(d6["interface"].to_numpy(), R6)                    # runs in round order, the R copies together
table_c["topology"] = np.repeat(d6["topology"].to_numpy(), R6)

ep = {"human": stuck.episode_table(runs_h, table_h, iface_h), "C2": stuck.episode_table(runs_c, table_c, iface_c)}
rounds = {"human": table_h.assign(interface=iface_h), "C2": table_c.assign(interface=iface_c)}
pd.DataFrame({w: r.groupby(["interface", "topology"]).size() for w, r in rounds.items()}).rename(
    columns={"human": "human rounds", "C2": "C2 rounds"})


human rounds  C2 rounds
interface topology                          
cc        Full                 34        480
          Hierarchy            36        500
          Star                 33        470
lab_v09   Full                 27        270
          Hierarchy            60        600

**How to read it.** Rounds per interface and topology: humans (G-train, bot-free, no old CL) and C2 (the G6train design without old CL, bot rounds simulated all-human, 10 replicates).

In [7]:
STATS = ["per_round", "duration_median", "first_try_median", "tries_median", "p_kept", "break_min_share"]
cells = []
for iface, topology in [("lab_v09", "Full"), ("lab_v09", "Hierarchy"), ("cc", "Full"), ("cc", "Hierarchy"), ("cc", "Star")]:
    for kind in stuck.TYPES[topology]:
        cell = {"interface": iface, "topology": topology, "type": kind}
        for who in ["human", "C2"]:
            e, r = ep[who], rounds[who]
            e = e[(e["interface"] == iface) & (e["topology"] == topology) & (e["type"] == kind)]
            r = r[(r["interface"] == iface) & (r["topology"] == topology)]
            if who == "human":
                for s, (v, lo, hi) in stuck.boot_cell(e, r).items():
                    cell[f"h_{s}"], cell[f"h_{s}_lo"], cell[f"h_{s}_hi"] = v, lo, hi
            else:
                cell.update({f"c_{s}": v for s, v in stuck.episode_stats(e, len(r)).items()})
        cells.append(cell)
cells = pd.DataFrame(cells)


def show_cell(r, s, dec):
    h = r[f"h_{s}"]
    if np.isnan(h):
        return f"— | {r[f'c_{s}']:.{dec}f}" if not np.isnan(r[f"c_{s}"]) else "—"
    c = r[f"c_{s}"]
    return f"{h:.{dec}f} [{r[f'h_{s}_lo']:.{dec}f}, {r[f'h_{s}_hi']:.{dec}f}] | " + ("—" if np.isnan(c) else f"{c:.{dec}f}")


DEC = {"per_round": 2, "duration_median": 1, "first_try_median": 2, "tries_median": 1, "p_kept": 2, "break_min_share": 2}
view = pd.DataFrame({"cell": cells["interface"].str.replace("lab_", "") + " " + cells["topology"] + " " + cells["type"],
                     "episodes h | C2": cells["h_episodes"].astype(int).astype(str) + " | " + cells["c_episodes"].astype(int).astype(str),
                     **{s: [show_cell(r, s, DEC[s]) if (s != "break_min_share" or r["topology"] == "Full") else ""
                            for _, r in cells.iterrows()] for s in STATS}})
print(view.to_string(index=False))


                     cell episodes h | C2                per_round        duration_median         first_try_median         tries_median                   p_kept          break_min_share
             v09 Full 6-1        12 | 557 0.44 [0.19, 0.74] | 2.06 10.8 [8.0, 20.6] | 4.9 1.55 [1.20, 2.30] | 1.70 4.0 [2.5, 7.0] | 2.0 0.73 [0.58, 0.82] | 0.76 0.18 [0.00, 0.40] | 0.17
             v09 Full 5-2         6 | 810 0.22 [0.04, 0.48] | 3.00   4.5 [3.3, 7.0] | 3.9 0.75 [0.60, 1.30] | 1.10 2.5 [2.0, 4.0] | 3.0 0.82 [0.75, 0.86] | 0.77 0.50 [0.33, 1.00] | 0.36
             v09 Full 4-3         0 | 386 0.00 [0.00, 0.00] | 1.43                — | 3.1                 — | 0.80              — | 3.0                 — | 0.79                 — | 0.46
    v09 Hierarchy fraught        70 | 714 1.17 [0.53, 1.97] | 1.19   6.3 [4.5, 8.8] | 5.3 1.21 [1.00, 1.70] | 1.80 2.0 [2.0, 2.0] | 2.0 0.79 [0.73, 0.83] | 0.73                         
v09 Hierarchy non-fraught       54 | 1945 0.90 [0.47, 1.42] | 3.24   4

**How to read it.** One row per cell (interface, topology, episode type). Each entry is human value [95% CI] | C2 value; C2 pools 10 replicates. per_round = episodes per round; durations and times in s; tries_median = tries until the episode ends; p_kept = kept / (kept + probes) per try (detector measure, same on both); break_min_share = share of the broken Full episodes broken by the minority. — = no episode (or no try) in that cell.

In [8]:
c = cells
first_ok = c["h_first_try_median_lo"].notna()
c["rule_i"] = first_ok & (c["c_first_try_median"] > 1.5 * c["h_first_try_median"]) & (
    (c["c_first_try_median"] < c["h_first_try_median_lo"]) | (c["c_first_try_median"] > c["h_first_try_median_hi"]))
c["rule_ii"] = c["c_tries_median"] > 1.5 * c["h_tries_median"]
types_i = sorted(set(c.loc[c["rule_i"], "type"]))
full = c["topology"] == "Full"
fires_i = len(types_i) >= 2
fires_ii = bool(c.loc[full, "rule_ii"].any() and not c.loc[~full, "rule_ii"].any())
print(c[["interface", "topology", "type", "h_first_try_median", "c_first_try_median", "rule_i", "h_tries_median",
         "c_tries_median", "rule_ii"]].round(2).to_string(index=False))
print()
print(f"(i)  time to first try > 1.5 x human and outside the CI: types {types_i or 'none'} -> "
      f"{'FIRES' if fires_i else 'does not fire'} (needs >= 2 of 5 types)")
if not fires_i:
    print(f"(ii) tries until the end > 1.5 x human: Full cells {c.loc[full & c['rule_ii'], 'type'].tolist()}, "
          f"other cells {(c.loc[~full & c['rule_ii'], 'interface'] + ' ' + c.loc[~full & c['rule_ii'], 'topology'] + ' ' + c.loc[~full & c['rule_ii'], 'type']).tolist()} -> "
          f"{'FIRES' if fires_ii else 'does not fire'}")
verdict = ("(i) candidate fix: a stuck feature in the try hazard" if fires_i else
           "(ii) candidate fix: standoff-specific keep values" if fires_ii else "(iii) freeze C2")
print("VERDICT:", verdict)


interface  topology        type  h_first_try_median  c_first_try_median  rule_i  h_tries_median  c_tries_median  rule_ii
  lab_v09      Full         6-1                1.55                1.70   False             4.0             2.0    False
  lab_v09      Full         5-2                0.75                1.10   False             2.5             3.0    False
  lab_v09      Full         4-3                 NaN                0.80   False             NaN             3.0    False
  lab_v09 Hierarchy     fraught                1.20                1.80   False             2.0             2.0    False
  lab_v09 Hierarchy non-fraught                0.84                1.10   False             2.0             2.0    False
       cc      Full         6-1                1.81                2.24   False             3.0             2.0    False
       cc      Full         5-2                0.99                1.35   False             2.0             2.0    False
       cc      Full         4-3 

**How to read it.** The reading rule of entry 4, cell by cell: `rule_i` = C2's median time to first try is > 1.5 x the human median and outside the human CI; `rule_ii` = C2's median tries until the end is > 1.5 x the human median. The verdict follows the rule's order: (i), else (ii), else (iii).

**What it shows (descriptive only; C2 is EXPLORATORY).**
- **Part 1.** Without old CL, the human CL targets move (e.g. Full CL success 82.5 -> 89.7%, KM median 89 -> 72 s; Hierarchy CL 10a 1.8 -> 1.7 s with a CI of [1.70, 1.85]). The models move more: C2 Full CL success 51.8 -> 77.7%, Hierarchy CL 72.5 -> 97.5% (now a pass). Passes over the 53: L1 17 -> 17, final 16 -> 16, C2 31 -> 30 (CL cells 8 -> 7); median |gap| L1 1.80 -> 1.60, final 1.86 -> 1.69, C2 0.74 -> 0.86. C2 still misses consensus time in both CL cells (150 vs. 72 s, 63 vs. 37 s) and is now too committed there (11.5 vs. 5.1%, 14.1 vs. 9.6%).
- **Part 2.** Once an episode starts, C2 resolves it like the humans: time to first try within 1.5 x the human median in every cell (the largest ratio is v09 Hierarchy fraught, 1.80 vs. 1.21 s, 1.49 x), the same median tries (2-3), P(kept) 0.70-0.81 vs. 0.61-1.00, and a Full minority-break share near the human one. What differs is how often episodes start: C2 has about 2-4 x the human episodes per round in most cells (e.g. v09 Hierarchy non-fraught 3.24 vs. 0.90, CC Full 5-2 4.06 vs. 0.62). The reading rule does not look at that.
- **Verdict (entry 4 rule): (iii) freeze C2.**
